# Contradictory, My Dear Watson — Multilingual NLI

End-to-end notebook for the [Kaggle competition](https://www.kaggle.com/competitions/contradictory-my-dear-watson):
given a (premise, hypothesis) pair in one of 15 languages, classify it as
**entailment (0) / neutral (1) / contradiction (2)**.

Pipeline: `src/` modules (data, model, training) wired together below.
Run cells top-to-bottom on a GPU runtime (~25 min on a Colab T4).

## 1. Setup

In [ ]:
# Make `src/` importable when the notebook lives in `notebooks/`.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"gpu:    {torch.cuda.get_device_name(0)}")

## 2. Load data

In [ ]:
from src.data import LABEL_NAMES, load_csvs

train_df, test_df = load_csvs()
print("train:", train_df.shape, "|", "test:", test_df.shape)
train_df.head()

## 3. Visualize

Sanity-check class balance and text length before training.

In [ ]:
import plotly.express as px

px.pie(
    train_df.groupby("language").size().reset_index(name="count"),
    names="language", values="count",
    title="train: language distribution",
)

In [ ]:
label_counts = (
    train_df["label"].value_counts().sort_index().rename(LABEL_NAMES)
)
px.bar(
    label_counts.reset_index().rename(
        columns={"index": "label", "label": "count"}
    ),
    x="label", y="count", title="train: label distribution",
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_df["premise"].str.len(), kde=True, ax=axes[0])
axes[0].set_title("premise length")
sns.histplot(train_df["hypothesis"].str.len(), kde=True, ax=axes[1])
axes[1].set_title("hypothesis length")
plt.tight_layout(); plt.show()

## 4. Train

`train_and_predict` does train/val split, tokenization, fine-tuning and
writes a Kaggle submission CSV. See `src/train.py` for the implementation.

In [ ]:
from src.train import TrainConfig, train_and_predict

cfg = TrainConfig(
    epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    seed=42,
)
submission_path = train_and_predict(cfg)
print(f"wrote: {submission_path}")

## 5. Inspect submission

In [ ]:
submission = pd.read_csv(submission_path)
print(submission.shape)
submission.head()